# temporal predictions 

Minimal notebook setup for exploring predictive coding with NumPy, PyTorch, and torchvision.

In [114]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import torch.nn.functional as F
import math
import os as os

In [115]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch: {torch.__version__}")
print(f"NumPy: {np.__version__}")

Using device: cpu
PyTorch: 2.13.0
NumPy: 2.5.2


dataset/input variables

In [116]:
if not os.path.exists("./data/MovingMNIST/mnist_test_seq.npy"):
    datasets.MovingMNIST._URL = "https://huggingface.co/datasets/goodgoodideas/STDN/resolve/main/mnist_test_seq.npy"
input_data = datasets.MovingMNIST(root="./data", split=None, download=True)

In [117]:
def make_variables(image, initial_r_size=64, num_channels=16, num_layers=2, kernel_size=4, padding=1, stride=2):
    I = image
    if I.ndim == 4:
        previous_num_channels = I.shape[1]
    elif I.ndim == 3:
        previous_num_channels = I.shape[0]
    else:
        previous_num_channels = 1
    r = []
    deconvs = []
    out_channels = num_channels * 2
    for i in range(num_layers):
        r_size = initial_r_size // (2 ** (i + 1))
        r.append(torch.randn(1, out_channels, r_size, r_size) * 0.01)
        deconvs.append(nn.ConvTranspose2d(out_channels, previous_num_channels, kernel_size=kernel_size, padding=padding, stride=stride, bias=False))
        previous_num_channels = out_channels
        out_channels = out_channels * 2
    return I, r, deconvs

temporal nn

In [118]:
class TemporalNN(nn.Module):
    def __init__(self, r, hidden_dim=256):
        super().__init__()
        self.mlps = nn.ModuleList()
        for ri in r:
            feat_dim = ri[0].numel()
            mlp = nn.Sequential(
                nn.Linear(2 * feat_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, feat_dim),
            )
            nn.init.zeros_(mlp[-1].weight)
            nn.init.zeros_(mlp[-1].bias)
            self.mlps.append(mlp)

    def forward(self, r_prev2, r_prev1):
        r_hat = []
        for mlp, r2, r1 in zip(self.mlps, r_prev2, r_prev1):
            x = torch.cat([r2.reshape(r2.shape[0], -1), r1.reshape(r1.shape[0], -1)], dim=-1)
            delta = mlp(x).view_as(r1)
            r_hat.append(r1 + delta)
        return r_hat


Training Loop

In [119]:


def train_loop(I_curr, r_prev2, r_prev1, r_curr, alpha, lambda_u, lr_r, lr_u,sigma_2, num_epochs_outer, num_epochs_inner, num_layers, deconvs, temporal_nn):
    I_curr = I_curr.float()
    if I_curr.ndim == 3:
        I_curr = I_curr.unsqueeze(0)
    e_spatial = [None] * num_layers
    e_temporal = [None] * num_layers
    f = lambda x: torch.clamp(x, min=-1.0, max=1.0)

    if r_prev1 is None:
        r_prev1 = [torch.zeros_like(ri) for ri in r_curr]
    else:
        r_prev1 = [ri.detach() for ri in r_prev1]

    if r_prev2 is None:
        r_prev2 = [torch.zeros_like(ri) for ri in r_prev1]
    else:
        r_prev2 = [ri.detach() for ri in r_prev2]

    
    for out_epoch in range(num_epochs_outer):
        r_prev1 = [ri.detach() for ri in r_prev1]
        r_prev2 = [ri.detach() for ri in r_prev2]
        r_pred = temporal_nn(r_prev2, r_prev1)

        for in_epoch in range(num_epochs_inner):
            e_spatial[0] = I_curr - f(deconvs[0](r_curr[0]))

            for i in range(1, num_layers):
                e_spatial[i] = r_curr[i - 1] - f(deconvs[i](r_curr[i]))
            
            for i in range(num_layers):
                cauchy_prior = alpha * (2 * r_curr[i] / (1 + r_curr[i] ** 2))
                bottom_up = F.conv2d(e_spatial[i], deconvs[i].weight, padding=deconvs[i].padding, stride=deconvs[i].stride)
                dr = (1.0 / sigma_2) * bottom_up - cauchy_prior - (1.0 / sigma_2) * (r_curr[i] - r_pred[i])
                if i < num_layers - 1:
                    dr = dr - (1.0 / sigma_2) * e_spatial[i + 1]
                r_curr[i] = (r_curr[i] + lr_r * dr).detach()
                r_curr[i].requires_grad = True

        e_spatial[0] = I_curr - f(deconvs[0](r_curr[0]))
        r_pred = temporal_nn(r_prev2, r_prev1)
        total_loss = 0

        for j in range(num_layers):
            if j > 0:
                e_spatial[j] = r_curr[j - 1] - f(deconvs[j](r_curr[j]))
            e_temporal[j] = r_curr[j] - r_pred[j]

            total_loss += (1.0 / (2 * sigma_2)) * (torch.sum(e_spatial[j]**2) + torch.sum(e_temporal[j]**2))

            deconvs[j].zero_grad()
        temporal_nn.zero_grad()
        
        total_loss.backward()
        
        
        
    
        with torch.no_grad():
            for j in range(num_layers):
                if deconvs[j].weight.grad is not None:
                    deconvs[j].weight -= lr_u * (deconvs[j].weight.grad + lambda_u * deconvs[j].weight)
                    deconvs[j].weight /= (torch.linalg.vector_norm(deconvs[j].weight, ord=2, dim=(1, 2, 3), keepdim=True) + 1e-8)
                    deconvs[j].weight.grad = None
            for param in temporal_nn.parameters():
                if param.grad is not None:
                    param -= lr_u * (param.grad + lambda_u * param)
                    param /= (torch.linalg.vector_norm(param, ord=2) + 1e-8)
                    param.grad = None

    return r_curr


sequential training

In [120]:
def train_video_sequence(sequence, r_curr_init, alpha, lambda_u, lr_r, lr_u, sigma_2, num_epochs_outer, num_epochs_inner, num_layers, deconvs, temporal_nn):
    r_prev2 = None
    r_prev1 = None
    r_curr = r_curr_init
    r_sequence = []
    
    for i, image in enumerate(sequence):
        I_curr = image.float()

        r_curr = train_loop(
            I_curr, r_prev2, r_prev1, r_curr, 
            alpha, lambda_u, lr_r, lr_u, sigma_2, 
            num_epochs_outer, num_epochs_inner, num_layers, 
            deconvs, temporal_nn
        )

        r_sequence.append([ri.detach().clone() for ri in r_curr])
        
        r_prev2 = r_prev1
        r_prev1 = [ri.detach() for ri in r_curr]   

    return r_sequence

validation


In [121]:
def validate_heirarchical(seq, r_init, num_epochs_inner, num_layers, sigma_2, alpha, lr_r, deconvs, temporal_nn, T_context=10, T_future=10):
    seq = seq.float()
    if seq.ndim == 3:
        seq = seq.unsqueeze(1)
    f = lambda x: torch.clamp(x, min=-1.0, max=1.0)

    r_prev2, r_prev1 = test(
        seq, r_init, num_epochs_inner, num_layers, sigma_2, alpha, lr_r,
        deconvs, temporal_nn, T_context, 0
    )

    future_true = []
    future_pred = []

    with torch.no_grad():
        for t in range(T_future):
            r_pred = temporal_nn(r_prev2, r_prev1)
            I_hat = f(deconvs[0](r_pred[0]))

            I_true = seq[T_context + t] - seq[T_context + t].mean()
            if I_true.ndim == 3:
                I_true = I_true.unsqueeze(0)

            future_true.append(I_true)
            future_pred.append(I_hat)

            r_prev2 = [ri.detach().clone() for ri in r_prev1]
            r_prev1 = [ri.detach().clone() for ri in r_pred]

    future_true = torch.stack(future_true, dim=0)
    future_pred = torch.stack(future_pred, dim=0)
    mse_rollout = torch.mean((future_true - future_pred) ** 2)
    mse_per_frame = torch.mean((future_true - future_pred) ** 2, dim=(1, 2, 3, 4))

    return mse_rollout, mse_per_frame, future_true, future_pred, r_prev1


test it

In [122]:
def test(seq, r_init, num_epochs_inner, num_layers, sigma_2, alpha, lr_r, deconvs, temporal_nn, T_context=10, T_future=10):
    seq = seq.float()
    if seq.ndim == 3:
        seq = seq.unsqueeze(1)
    f = lambda x: torch.clamp(x, min=-1.0, max=1.0)
    e_spatial = [None] * num_layers

    r_prev2 = None
    r_prev1 = None
    r_curr = [ri.clone() for ri in r_init]
    for t in range(T_context):
        I_curr = seq[t] - seq[t].mean()
        if I_curr.ndim == 3:
            I_curr = I_curr.unsqueeze(0)
        if r_prev1 is None:
            r_prev1_use = [torch.zeros_like(ri) for ri in r_curr]
            r_prev2_use = [torch.zeros_like(ri) for ri in r_curr]
        elif r_prev2 is None:
            r_prev1_use = [ri.detach() for ri in r_prev1]
            r_prev2_use = [torch.zeros_like(ri) for ri in r_curr]
        else:
            r_prev1_use = [ri.detach() for ri in r_prev1]
            r_prev2_use = [ri.detach() for ri in r_prev2]
        with torch.no_grad():
            r_pred = temporal_nn(r_prev2_use, r_prev1_use)
            for _ in range(num_epochs_inner):
                e_spatial[0] = I_curr - f(deconvs[0](r_curr[0]))
                for i in range(1, num_layers):
                    e_spatial[i] = r_curr[i - 1] - f(deconvs[i](r_curr[i]))
                for i in range(num_layers):
                    cauchy_prior = alpha * (2 * r_curr[i] / (1 + r_curr[i] ** 2))
                    bottom_up = F.conv2d(e_spatial[i], deconvs[i].weight, padding=deconvs[i].padding, stride=deconvs[i].stride)
                    dr = (1.0 / sigma_2) * bottom_up - cauchy_prior - (1.0 / sigma_2) * (r_curr[i] - r_pred[i])
                    if i < num_layers - 1:
                        dr = dr - (1.0 / sigma_2) * e_spatial[i + 1]
                    r_curr[i] = (r_curr[i] + lr_r * dr).detach()
        r_prev2 = r_prev1
        r_prev1 = [ri.detach().clone() for ri in r_curr]

    if T_future == 0:
        if r_prev2 is None:
            r_prev2 = [torch.zeros_like(ri) for ri in r_prev1]
        return r_prev2, r_prev1

    future_pred = []
    with torch.no_grad():
        for t in range(T_future):
            r_pred = temporal_nn(r_prev2, r_prev1)
            I_hat = f(deconvs[0](r_pred[0]))
            future_pred.append(I_hat)
            r_prev2 = [ri.detach().clone() for ri in r_prev1]
            r_prev1 = [ri.detach().clone() for ri in r_pred]

    future_pred = torch.stack(future_pred, dim=0)
    return r_prev1, future_pred


In [ ]:
def main():
    num_data = 20
    T_context = 10
    T_future = 10
    sequences = [input_data[i].float() / 255.0 for i in range(num_data)]
    num_train_epochs = 20
    alpha = 0.001
    lambda_u = 0.001
    lr_r = 0.005
    lr_U = 0.002
    sigma_2 = 1.0
    num_layers = 2

    num_epochs_inner = 50
    num_epochs_outer = 1
    f = lambda x: torch.clamp(x, min=-1.0, max=1.0)

    print("=" * 60)
    print(f"Starting Training: {num_train_epochs} Epochs | {num_data} Sequences | {num_layers} Layers")
    print(f"Hyperparameters: lr_r={lr_r}, lr_U={lr_U}, alpha={alpha}, lambda_u={lambda_u}")
    print("=" * 60)

    _, r_init, deconvs = make_variables(sequences[0][0], num_layers=num_layers)
    temporal_nn = TemporalNN(r_init)

    for epoch in range(num_train_epochs):
        for seq in sequences:
            r_prev2 = None
            r_prev1 = None
            r_curr = [ri.clone() for ri in r_init]
            for t in range(T_context):
                I = seq[t] - seq[t].mean()
                r_curr = train_loop(I, r_prev2, r_prev1, r_curr, alpha, lambda_u, lr_r, lr_U, sigma_2, num_epochs_outer, num_epochs_inner, num_layers, deconvs, temporal_nn)
                r_prev2 = r_prev1
                r_prev1 = [ri.detach().clone() for ri in r_curr]

        if epoch % 5 == 0 or epoch == num_train_epochs - 1:
            mse_rollout, mse_per_frame, future_true, future_pred, _ = validate_heirarchical(
                sequences[0], r_init, num_epochs_inner, num_layers, sigma_2, alpha, lr_r, deconvs, temporal_nn, T_context, T_future
            )
            print(f"[Epoch {epoch:03d}/{num_train_epochs:03d}]  Rollout MSE: {mse_rollout:.6f}  |  Per-frame: {[f'{x:.4f}' for x in mse_per_frame.tolist()]}")

            fig, axes = plt.subplots(2, T_future, figsize=(2 * T_future, 4))
            for t in range(T_future):
                axes[0, t].imshow(future_true[t].reshape(64, 64).detach().numpy(), cmap="gray", vmin=-0.5, vmax=0.5)
                axes[0, t].set_title(f"True {T_context + t}")
                axes[0, t].axis("off")
                axes[1, t].imshow(future_pred[t].reshape(64, 64).detach().numpy(), cmap="gray", vmin=-0.5, vmax=0.5)
                axes[1, t].set_title(f"Pred {T_context + t}")
                axes[1, t].axis("off")
            plt.show()

    val_seq = input_data[num_data].float() / 255.0
    mse_rollout, mse_per_frame, future_true, future_pred, _ = validate_heirarchical(
        val_seq, r_init, num_epochs_inner, num_layers, sigma_2, alpha, lr_r, deconvs, temporal_nn, T_context, T_future
    )
    print(f"Validation  ->  Rollout MSE: {mse_rollout:.6f}  |  Per-frame: {[f'{x:.4f}' for x in mse_per_frame.tolist()]}")

    fig, axes = plt.subplots(2, T_future, figsize=(2 * T_future, 4))
    for t in range(T_future):
        axes[0, t].imshow(future_true[t].reshape(64, 64).detach().numpy(), cmap="gray", vmin=-0.5, vmax=0.5)
        axes[0, t].set_title(f"True {T_context + t}")
        axes[0, t].axis("off")
        axes[1, t].imshow(future_pred[t].reshape(64, 64).detach().numpy(), cmap="gray", vmin=-0.5, vmax=0.5)
        axes[1, t].set_title(f"Pred {T_context + t}")
        axes[1, t].axis("off")
    plt.show()

if __name__ == "__main__":
    main()


Starting Training: 20 Epochs | 20 Sequences | 2 Layers
Hyperparameters: lr_r=0.005, lr_U=0.002, alpha=0.001, lambda_u=0.001
